In [ ]:
import os, sys
sys.path.append(os.path.abspath("..")) 
from src.state_schema import SupervisorState
from src.supervisor_graph import supervisor_node
from src.complaint_graph import complaint_flow
from src.inquiry_graph import inquiry_flow
from src.retention_graph import retention_flow
from src.conversation_graph import greeting_flow, clarification_flow
from rich import print as rprint

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
def build_graph():

    graph = StateGraph(SupervisorState)

    graph.add_node("supervisor",supervisor_node)
    graph.add_node("greeting_flow",greeting_flow)
    graph.add_node("clarification_flow", clarification_flow)
    graph.add_node("complaint_flow", complaint_flow)
    graph.add_node("retention_flow", retention_flow)
    graph.add_node("inquiry_flow", inquiry_flow)

    graph.add_edge(START, "supervisor")
    graph.add_edge("greeting_flow", END)
    graph.add_edge("clarification_flow", END)

    memory = MemorySaver()
    return graph.compile(checkpointer=memory)

graph = build_graph()


In [ ]:
graph

In [ ]:
def agent(state:SupervisorState, thread_id:str)->SupervisorState:
    config = {"configurable": {"thread_id": thread_id}}
    return graph.invoke(state,config)

In [ ]:
test_cases = [
    # --- FLOW 1: GREETINGS & CHITCHAT ---
    {"user_input": "Hello! Is there anyone available to help me with a few questions?"},
    {"user_input": "Good morning, I hope you're having a productive day."},

    # --- FLOW 2: TECHNICAL COMPLAINTS (Complaint Graph) ---
    {"user_input": "My home internet has been dropping every 20 minutes since the rain started."},
    {"user_input": "The screen on my new device is flickering and I can't see the menu."},
    {"user_input": "I've already tried restarting the router, but the red light is still blinking."},

    # --- FLOW 3: ACCOUNT & RETENTION (Retention Graph) ---
    {"user_input": "I want to cancel my subscription effective immediately; it's too expensive."},
    {"user_input": "My contract is up next month and I'm looking at switching to a different provider."},
    {"user_input": "I was promised a discount on my last bill that never showed up. I'm very upset."},

    # --- FLOW 4: GENERAL INQUIRIES (Inquiry Graph) ---
    {"user_input": "How do I set up international roaming for my upcoming trip to Europe?"},
    {"user_input": "What are your store hours for the downtown St. Louis location?"},
    {"user_input": "Can you explain the difference between the Basic and Pro service plans?"},

    # --- FLOW 5: AMBIGUOUS (Clarification Node) ---
    {"user_input": "I'm having a really hard time with this service and I need a solution."},
    {"user_input": "This is the third time I'm reaching out about my order status."},
    {"user_input": "I'm not sure if I'm in the right place, but I need help with my account."},

    # --- MULTI-TURN / STICKY FLOW TESTS (Requires same thread_id) ---
    {"user_input": "My phone won't turn on."}, # Turn 1: Should enter Complaint
    {"user_input": "Yes, I held the power button for 10 seconds like you said."}, # Turn 2: Should stay in Complaint
    {"user_input": "Wait, actually, how much would it cost just to upgrade to a new phone instead?"} # Turn 3: Should pivot or reset
]

In [ ]:
res = agent({"user_input":
            #  "Keyboard has issue."})
            "Its iPhone15"})

In [ ]:
res = agent(test_cases[3], '7')

In [ ]:
test_cases[5]

In [ ]:
for msg in res['messages']:
    rprint(msg.content)

In [ ]:
import json

In [ ]:
data = {   
    "message": res.get("messages")[-1].content,
    "extracted_info": res.get("extracted_info"),
    "complaint_data": res.get("complaint_data"),
    "missing_info": res.get("missing_info"),
    "active_flow": res.get("active_flow"),
    "customer_profile": res.get("customer_profile"),
}

In [ ]:
data

In [ ]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer

# model_name = "cross-encoder/nli-deberta-v3-small"
# save_path = "/Volumes/LaCie/Projects_portfolio/GenAI/Agentic User Resolution Assistant (AURA)/models/deberta-small"

# print(f"Downloading {model_name}...")

# # Download and save the core components
# model = AutoModelForSequenceClassification.from_pretrained(model_name)
# tokenizer = AutoTokenizer.from_pretrained(model_name)

# model.save_pretrained(save_path)
# tokenizer.save_pretrained(save_path)

# print(f"Successfully saved to {save_path}")